# HLS and MODIS Comparison
---
**Summary**
The following code compiles the data from MODIS and HLS into their own respective DataFrames that contain a row for each composite as well as the vegatation index mean, median, min, max, standard deviation, and valid pixels. These DataFrames are then used to create figures comparing the two, including a vegetation index time series, a graph of the vegetation index multi-year seasonal means, and an annual phenology summary that shows the annual min, max, start of season (calculated as the first DOY to reach the greenup threshold, 15% amplitude), end of season (first DOY to fall below threshold after peak), and season length.

## Preparing the Data

In [ ]:
# Import packages
from phenology_algorithm import *

In [ ]:
# reload modules during block executions for active dev
# in phenometrics_algos.py and phenometrics_utils.py
# %reload_ext autoreload
%load_ext autoreload
%autoreload 2

In [ ]:
# Initialize variables
veg_index = "evi2"
tile_id = "15TYL"
output_dir = f"hls_modis_comparisons/{tile_id}"
os.makedirs(output_dir, exist_ok=True)
start_year = 2016
end_year = 2025

In [ ]:
modis, modis_vi, modis_doy = compute_modis_vi_stats(veg_index,
                               f'./modis_veg_indices/{tile_id}/modis_tiff/MOD13Q1/', 
                               outdir = output_dir,
                               tile_id = tile_id,
                               start_year = start_year,
                               end_year = end_year,
                               hls_data_dir=f'./hls_veg_indices/{tile_id}/',
                               # roi_name = "betampona",
                               # roi_shp='./roi_shp/betampona.zip'
                               # roi_name = "cartercountry",
                               # roi_shp='./roi_shp/cartercountry.zip'
                               # roi_name = "patuxent",
                               # roi_shp='./roi_shp/patuxent.zip'
                               # roi_name = "dangermond",
                               # roi_shp='./roi_shp/dangermond.zip'
                               # roi_name = "bci",
                               # roi_shp='./roi_shp/bci.zip'
                               roi_name = "oneida",
                               roi_shp='./roi_shp/oneida.zip'
)

In [ ]:
hls, hls_vi, hls_doy = compute_hls_vi_stats(veg_index, 
                           f'./hls_veg_indices/{tile_id}/', 
                           outdir = output_dir,              
                           tile_id = tile_id,
                           start_year = start_year, 
                           end_year = end_year,
                           # roi_name = "betampona",
                           # roi_shp='./roi_shp/betampona.zip'
                           # roi_name = "cartercountry",
                           # roi_shp='./roi_shp/cartercountry.zip'
                           # roi_name = "patuxent",
                           # roi_shp='./roi_shp/patuxent.zip'
                           # roi_name = "dangermond",
                           # roi_shp='./roi_shp/dangermond.zip'
                           # roi_name = "bci",
                           # roi_shp='./roi_shp/bci.zip'
                           roi_name = "oneida",
                           roi_shp='./roi_shp/oneida.zip'
)

## Timeseries

In [ ]:
plot_vi_timeseries(hls, modis, veg_index, output_dir, tile_id=tile_id, vi_min=None)

In [ ]:
plot_vi_seasonal_mean(hls, modis, veg_index, output_dir, tile_id=tile_id, vi_min=None)

## Phenometrics

In [ ]:
phenometrics(hls_vi,
             hls_doy,
             "HLS",
             veg_index,
             outdir = output_dir,
             tile_id = tile_id,
             start_year = start_year,
             end_year = end_year)

In [ ]:
phenometrics(modis_vi,
             modis_doy,
             "MODIS",
             veg_index,
             outdir = output_dir,
             tile_id = tile_id,
             start_year = start_year,
             end_year = end_year)

## Single Test Mode

In [10]:
from phenometrics_utils import *
from functools import partial

In [11]:
data_dir=Path("./hls_veg_indices/")
# Path("./modis_veg_indices/15TYL/modis_tiff/MOD13Q1/")
output_path=Path("./output/")
tile="15TYL" #"03WWM-subset-council"
target_year=2025
context_months=12 
chunk_size=250
n_workers=1
roi_file=None
tile_epsg=None
chunks_in_memory=10
run_label=None

In [12]:
# Output directory
out_subdir = f"{tile}"
if run_label:
    out_subdir = f"{out_subdir}-{run_label}"
if roi_file is not None:
    out_subdir = f"{out_subdir}-{roi_file.parent.name}"

output_dir = output_path / out_subdir / str(target_year)
output_dir.mkdir(parents=True, exist_ok=True)

In [13]:
print("=" * 70)
print(f"  Tile         : {tile}")
print(f"  Target year  : {target_year}")
print(f"  Input dir    : {data_dir}")
print(f"  Output dir   : {output_dir}")
print("=" * 70)

data_config = ProcessingConfig(
    base_path=Path(data_dir),
    tile_id=tile
)
scenes = build_scene_index(data_config)
available_years = sorted({s.year for s in scenes})
if target_year not in available_years:
    raise ValueError(
        f"target_year={target_year} has no EVI scenes in local directory {data_dir}. "
        f"Available years: {available_years}"
    )

roi_reproj = None
if roi_file is not None:
    roi = gpd.read_file(roi_file)
    roi_reproj = roi.to_crs(f"EPSG:{tile_epsg}")
    print(f"  ROI loaded & reprojected to EPSG:{tile_epsg}")

reader = ChunkedTimeSeriesReaderStreaming(
    scenes,
    chunk_size=(chunk_size, chunk_size),
    roi=roi_reproj,
    duplicate_handling="mean",
    output_dir=output_dir,
    context_months=context_months,
    target_year=target_year,
    default_crs=tile_epsg,
)
configured_pipeline = partial(
    full_pipeline_chunk,
    apply_threshold=True,
    testing_mode=True,
    fill_snow_gaps=False,
)

  Tile         : 15TYL
  Target year  : 2025
  Input dir    : hls_veg_indices
  Output dir   : output/15TYL/2025
hls_veg_indices/15TYL
Found 0 EVI files in hls_veg_indices/15TYL
No valid scenes found!
No scenes to save


ValueError: target_year=2025 has no EVI scenes in local directory hls_veg_indices. Available years: []

In [ ]:
reload_chunk = True
if reload_chunk == False:
    print("Not rerunning")
else:
    evi_da = reader.load_chunk(0)

In [ ]:
chunk_results = configured_pipeline(
    evi_da,
    target_year=target_year, 
    apply_threshold=True,
    n_jobs=1,
    _pool=None,
    testing_mode=True,
    fill_snow_gaps=False,
    smoother="savgol"
)

In [ ]:
intermediate = chunk_results.pop('_intermediate')

print("Pipeline stages available:")
for stage_name, stage_data in intermediate.items():
    if stage_data is not None:
        print(f"  {stage_name}: {stage_data.shape}")

In [ ]:
chunk_idx = 0 
if intermediate:
    # Get chunk coordinates
    y_slice, x_slice = reader.chunk_slices[chunk_idx]
    chunk_y = reader.y_coords[y_slice]
    chunk_x = reader.x_coords[x_slice]
   
    # Check which variables are available
    if 'post_despike' in intermediate and 'post_spline' in intermediate:
        if intermediate['post_snow_fill'] is not None:
            nc_file = save_spline_comparison_netcdf(
                pre_spline=intermediate['post_despike'],
                post_spline=intermediate['post_snow_fill'],
                central_year=target_year,
                output_dir=output_dir,
                y_coords=chunk_y,
                x_coords=chunk_x,
                crs=str(reader.crs)
            )
        else:
            nc_file = save_spline_comparison_netcdf(
                pre_spline=intermediate['post_despike'],
                post_spline=intermediate['post_spline'],
                central_year=target_year,
                output_dir=output_dir,
                y_coords=chunk_y,
                x_coords=chunk_x,
                crs=str(reader.crs)
            )
        print(f"\n Saved to {nc_file}")
        
        # Optionally view it immediately
        ds = xr.open_dataset(nc_file)
        print("\nDataset contents:")
        print(ds)

In [ ]:
plot_spline_comparison(ds, yi=5, xi=0, target_year=target_year, outdir=output_dir)

In [ ]:
# Usage
plot_pixel_phenometrics(ds, chunk_results, yi=5, xi=0, target_year=target_year)

In [ ]:
intermediate.keys()

In [ ]:
# calc_obs_snow_background(ds)
x = 9
y = 9
evi_obs = intermediate['post_despike'].sel(time = str(target_year)).isel(y=y,x=x)
evi_post_spline = intermediate['post_spline'].sel(time = str(target_year)).isel(y=y,x=x)
if intermediate['post_snow_fill'] is not None:
    evi_post_snow_fill = intermediate['post_snow_fill'].sel(time = str(target_year)).isel(y=y,x=x)

In [ ]:
threshold = (evi_obs.min() + (evi_obs.max(dim='time', skipna=True) - evi_obs.min(dim='time', skipna=True)) * 0.05).values
threshold

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
# ax.axhline(y = threshold, label = f"Threshold:{threshold:.2f}")
ax.scatter(evi_obs.time, evi_obs.values, label = 'EVI Despiked', color = 'blue')
ax.plot(evi_post_spline.time, evi_post_spline.values, label = 'EVI Spline fit', linewidth = 2, color = 'green', linestyle = ':')
if intermediate['post_snow_fill'] is not None:
    ax.plot(evi_post_snow_fill.time, evi_post_snow_fill.values, label = 'EVI Snow filled',  linewidth = 1, color = 'orange')

# Add labels/titles
plt.suptitle(f"{tile}: x,y = {x},{y}")
plt.legend()
plt.gcf().autofmt_xdate() 
plt.tight_layout() # Prevents overlap
plt.show()

In [ ]:
evi_obs.time

In [ ]:
evi_obs.values

In [ ]:
for metric, arr in chunk_results.items():
    if isinstance(arr, np.ndarray):
        if arr.ndim == 3:
            val = arr[0, 0, 1]  # (year, y, x)
        elif arr.ndim == 2:
            val = arr[0, 1]     # (y, x)
        else:
            val = arr
        print(f"{metric:<25} {val:.4f}" if not np.isnan(val) else f"{metric:<25} NaN")

In [ ]:
# Or plot spatial maps for a specific date
date_idx = 37
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ds['evi_raw'].isel(time=date_idx).plot(ax=axes[0], cmap='RdYlGn')
axes[0].set_title(f'Raw EVI - {ds.time.values[date_idx]}')

ds['evi_smoothed'].isel(time=date_idx).plot(ax=axes[1], cmap='RdYlGn')
axes[1].set_title('Smoothed EVI')

ds['difference'].isel(time=date_idx).plot(ax=axes[2], cmap='RdBu_r', center=0)
axes[2].set_title('Difference')

plt.tight_layout()
# plt.savefig('spatial_comparison.png', dpi=150)

## Phenometric Plots

### Pixel Phenometrics

In [ ]:
for year in range(start_year, end_year+1):
    plot_annual_pixel_phenometrics(veg_index, "HLS", output_dir, tile_id, year)
    plot_annual_pixel_phenometrics(veg_index, "MODIS", output_dir, tile_id, year)

### Mean Phenometrics

In [ ]:
summary_df = build_summary_df_from_tifs(veg_index, output_dir, tile_id, start_year, end_year)
# print(summary_df)

In [ ]:
plot_phenology_summary(summary_df, veg_index, output_dir, tile_id)